PROJETO: Análise de Vendas Automotivas<br>
ETAPA: Tratamento Dos Dados<br>
ORIGEM: Dataset/Brutos<br>
DESTINO INICIAL: Dataset/Tratados

ETL — Transformação tabela de customers
- Tratamento de valores nulos
- Remoção de duplicidades
- Padronização de textos
- Correção de tipos de dados
- Normalização de categorias
- Validação de chaves
- Tratamento de inconsistências

In [ ]:
# Importando as bibliotecas
from pathlib import Path
import unicodedata
import re

In [ ]:
# Definindo o diretório base e os caminhos para armazenar os dados brutos e tratados.
BASE_DIR = Path.cwd().parent

RAW_DIR = BASE_DIR / 'Dataset' / 'brutos'
TREATED_DIR = BASE_DIR / 'Dataset' / 'tratados'

In [ ]:
# Carregando o arquivo CSV de clientes a partir do diretório de dados brutos.
df_customers = pd.read_csv(RAW_DIR / 'customers.csv')

In [60]:
display(df_customers.head(10))
df_customers.info()
df_customers.isnull().sum()

,customer_id,cpf,first_name,last_name,state,city,birth_date,income,score,professional_status,email,mobile
0,73650706FFAE,84932586604,IVONELIA,ARGULINO,SP,SÃO PAULO,1973-01-24,7000.0,491.0,clt,ivonelia.argulino265830@gmail.com,69262265830
1,236A739A6063,5912702104,ESTEVANIA,MARTHUS,SP,SÃO PAULO,1969-06-30,5800.0,376.0,clt,estevania.marthus204682@gmail.com,40632204682
2,CF2DFCBE74E3,85148454740,CLEUDINE,DUARTE,MG,BELO HORIZONTE,1945-12-25,3190.0,279.0,clt,cleudine.duarte661670@gmail.com,81639661670
3,25504A496AEB,12430594512,ADEIZE,LUZILDO,SC,NAVEGANTES,1948-03-25,11000.0,309.0,civil_servant,adeize.luzildo797118@gmail.com,21943797118
4,05DE6BCDFEC7,1571615400,SHEYLLA,JESKA,SP,CARAPICUÍBA,1963-10-31,6000.0,305.0,clt,sheylla.jeska608366@gmail.com,65148608366
5,CF65A8E9C692,12191189380,NACLECIA,GENAI,SP,SÃO PAULO,1959-10-22,6000.0,486.0,other,naclecia.genai578212@gmail.com,25706578212
6,AC89FF38B365,43696945874,JUVENIL,DHIOVANE,SC,GARUVA,1952-05-27,6500.0,564.0,other,juvenil.dhiovane446542@gmail.com,27845446542
7,1D4EF6B29AFF,99373641602,HARINA,DUARTE,SP,SERTÃOZINHO,1982-04-15,6000.0,600.0,clt,harina.duarte626566@gmail.com,63866626566
8,F21B059B9F17,54697671582,EDIVALDINA,LAURICIO,SP,PRAIA GRANDE,1987-07-10,6000.0,315.0,clt,edivaldina.lauricio861802@gmail.com,71908861802
9,DC3D147117FA,14154880400,AURETE,DUARTE,MG,BELO HORIZONTE,1998-06-13,5900.0,344.0,retired,aurete.duarte848990@gmail.com,87831848990


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25109 entries, 0 to 25108
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   customer_id          25109 non-null  object 
 1   cpf                  25109 non-null  int64  
 2   first_name           25109 non-null  object 
 3   last_name            25109 non-null  object 
 4   state                25109 non-null  object 
 5   city                 25109 non-null  object 
 6   birth_date           25109 non-null  object 
 7   income               25109 non-null  float64
 8   score                25109 non-null  float64
 9   professional_status  25109 non-null  object 
 10  email                25109 non-null  object 
 11  mobile               25109 non-null  int64  
dtypes: float64(2), int64(2), object(8)
memory usage: 2.3+ MB


customer_id            0
cpf                    0
first_name             0
last_name              0
state                  0
city                   0
birth_date             0
income                 0
score                  0
professional_status    0
email                  0
mobile                 0
dtype: int64

In [61]:
# Renomeando as colunas para português

df_customers.rename(columns={
    'customer_id': 'id_cliente',
    'first_name': 'primeiro_nome',
    'last_name': 'sobrenome',
    'state': 'estado',
    'city': 'cidade',
    'birth_date': 'data_nascimento',
    'income': 'renda',
    'score': 'pontuacao',
    'professional_status': 'status_profissional_original',
    'mobile': 'celular'
}, inplace=True)

In [62]:
df_customers

,id_cliente,cpf,primeiro_nome,sobrenome,estado,cidade,data_nascimento,renda,pontuacao,status_profissional_original,email,celular
0,73650706FFAE,84932586604,IVONELIA,ARGULINO,SP,SÃO PAULO,1973-01-24,7000.0,491.0,clt,ivonelia.argulino265830@gmail.com,69262265830
1,236A739A6063,5912702104,ESTEVANIA,MARTHUS,SP,SÃO PAULO,1969-06-30,5800.0,376.0,clt,estevania.marthus204682@gmail.com,40632204682
2,CF2DFCBE74E3,85148454740,CLEUDINE,DUARTE,MG,BELO HORIZONTE,1945-12-25,3190.0,279.0,clt,cleudine.duarte661670@gmail.com,81639661670
3,25504A496AEB,12430594512,ADEIZE,LUZILDO,SC,NAVEGANTES,1948-03-25,11000.0,309.0,civil_servant,adeize.luzildo797118@gmail.com,21943797118
4,05DE6BCDFEC7,1571615400,SHEYLLA,JESKA,SP,CARAPICUÍBA,1963-10-31,6000.0,305.0,clt,sheylla.jeska608366@gmail.com,65148608366
...,...,...,...,...,...,...,...,...,...,...,...,...
25104,2A627F45FEBE,29050871368,LAUDISEIA,DUARTE,SP,SOROCABA,1993-02-07,4200.0,446.0,clt,laudiseia.duarte723732@gmail.com,41697723732
25105,8186B0E0AD47,93191882060,FRANCILINO,DUARTE,SP,SÃO PAULO,1994-01-04,4500.0,442.0,clt,francilino.duarte240846@gmail.com,25681240846
25106,028608D204D3,56889314686,ADMILDA,DUARTE,MG,BELO HORIZONTE,1982-07-16,6500.0,388.0,self_employed,admilda.duarte476782@gmail.com,74256476782
25107,A5E64FD582D6,2741194304,EVANISE,MARTHUS,SP,EMBU DAS ARTES,1996-10-27,9500.0,915.0,self_employed,evanise.marthus651450@gmail.com,615651450


In [63]:
# Normalizar a coluna cidade para remover acentos
def remover_acentos(texto):
    if isinstance(texto, str):
        texto = unicodedata.normalize('NFKD', texto)
        return ''.join(
            caractere for caractere in texto
            if not unicodedata.combining(caractere)
        )
    return texto

# Aplicando a normalização
df_customers['cidade'] = (
    df_customers['cidade']
    .apply(remover_acentos)
    .str.upper()
    .str.strip()
)

In [64]:
df_customers['cidade']

0             SAO PAULO
1             SAO PAULO
2        BELO HORIZONTE
3            NAVEGANTES
4           CARAPICUIBA
              ...      
25104          SOROCABA
25105         SAO PAULO
25106    BELO HORIZONTE
25107    EMBU DAS ARTES
25108          SOROCABA
Name: cidade, Length: 25109, dtype: object

In [65]:
df_customers['status_profissional_original'].unique()

array(['clt', 'civil_servant', 'other', 'retired', 'businessman',
       'self_employed', 'freelancer', 'student'], dtype=object)

In [66]:
# Dicionário de tradução
traducao_status = {
    'clt': 'Empregado CLT',
    'civil_servant': 'Servidor público',
    'other': 'Outros',
    'retired': 'Aposentado',
    'businessman': 'Empresário',
    'self_employed': 'Trabalhador autônomo',
    'freelancer': 'Profissional freelancer',
    'student': 'Estudante'
}

# Criando nova coluna traduzida
df_customers['status_profissional'] = (
    df_customers['status_profissional_original']
    .str.strip()
    .str.lower()
    .map(traducao_status)
    .fillna('Não identificado')
)

In [67]:
df_customers[
    ['status_profissional_original', 'status_profissional']
].drop_duplicates().sort_values('status_profissional')

,status_profissional_original,status_profissional
9,retired,Aposentado
0,clt,Empregado CLT
11,businessman,Empresário
5761,student,Estudante
5,other,Outros
21,freelancer,Profissional freelancer
3,civil_servant,Servidor público
15,self_employed,Trabalhador autônomo


In [68]:
df_customers['status_profissional'].value_counts()

status_profissional
Empregado CLT              16307
Outros                      2271
Empresário                  1894
Trabalhador autônomo        1809
Profissional freelancer     1328
Aposentado                  1070
Servidor público             429
Estudante                      1
Name: count, dtype: int64

Transformação tabela de customers

In [39]:
df_ibge = pd.read_csv(RAW_DIR / 'ibge_genders.csv')
df_ibge

,first_name,gender
0,maiuly,female
1,jesuildo,male
2,geara,female
3,demilda,female
4,aleuda,female
...,...,...
100782,abimail,female
100783,seigi,male
100784,nadilma,female
100785,kathallen,female


In [40]:
df_ibge.rename(columns={
    'first_name': 'primeiro_nome',
    'gender': 'genero'
}, inplace=True)

In [41]:
df_ibge['primeiro_nome'] = (
    df_ibge['primeiro_nome']
    .str.strip()
    .str.upper()
)

In [43]:
traducao_genero = {
    'male': 'Masculino',
    'female': 'Feminino'
}

df_ibge['genero'] = (
    df_ibge['genero']
    .str.strip()
    .str.lower()
    .map(traducao_genero)
    .fillna('Não identificado')
)

In [44]:
df_ibge

,primeiro_nome,genero
0,MAIULY,Feminino
1,JESUILDO,Masculino
2,GEARA,Feminino
3,DEMILDA,Feminino
4,ALEUDA,Feminino
...,...,...
100782,ABIMAIL,Feminino
100783,SEIGI,Masculino
100784,NADILMA,Feminino
100785,KATHALLEN,Feminino


Transformação tabela de Products

In [45]:
df_products = pd.read_csv(RAW_DIR / 'products.csv')
df_products

,product_id,brand,model,model_year,price
0,produto001,RENAULT,LOGAN,2014,67500.0
1,produto002,VOLKSWAGEN,GOL,2012,51000.0
2,produto003,FIAT,PALIO,2016,61500.0
3,produto004,NISSAN,SENTRA,2012,53850.0
4,produto005,FIAT,UNO,2015,52500.0
...,...,...,...,...,...
328,produto329,CHERY,ARRIZO,2019,101850.0
329,produto330,MERCEDES-BENZ,C-300,2020,525000.0
330,produto331,FOTON,AUMARK,2015,165000.0
331,produto332,AUDI,TTS,2017,450000.0


In [46]:
df_products.rename(columns={
    'product_id': 'id_produto',
    'brand': 'marca',
    'model': 'modelo',
    'model_year': 'ano_modelo',
    'price': 'preco'
}, inplace=True)

In [47]:
def padronizar_texto(texto):
    if isinstance(texto, str):
        texto = texto.strip().upper()

        texto = unicodedata.normalize('NFKD', texto)

        texto = ''.join(
            c for c in texto
            if not unicodedata.combining(c)
        )

        texto = re.sub(r'[_-]+', ' ', texto)

        texto = re.sub(r'\s+', ' ', texto)

        return texto

    return texto

In [48]:
colunas_texto = ['marca', 'modelo']

for coluna in colunas_texto:
    df_products[coluna] = (
        df_products[coluna].apply(padronizar_texto)
    )

In [49]:
df_products

,id_produto,marca,modelo,ano_modelo,preco
0,produto001,RENAULT,LOGAN,2014,67500.0
1,produto002,VOLKSWAGEN,GOL,2012,51000.0
2,produto003,FIAT,PALIO,2016,61500.0
3,produto004,NISSAN,SENTRA,2012,53850.0
4,produto005,FIAT,UNO,2015,52500.0
...,...,...,...,...,...
328,produto329,CHERY,ARRIZO,2019,101850.0
329,produto330,MERCEDES BENZ,C 300,2020,525000.0
330,produto331,FOTON,AUMARK,2015,165000.0
331,produto332,AUDI,TTS,2017,450000.0


Transformação tabela de Regions

In [54]:
df_regions = pd.read_csv(RAW_DIR / 'regions.csv')
df_regions

,city,state,region,size,code,population
0,Alta Floresta Doeste,RO,Região Norte,Pequeno II,69,24392.0
1,Ariquemes,RO,Região Norte,Médio,69,90353.0
2,Cabixi,RO,Região Norte,Pequeno I,69,6313.0
3,Cacoal,RO,Região Norte,Médio,69,78574.0
4,Cerejeiras,RO,Região Norte,Pequeno I,69,17029.0
...,...,...,...,...,...,...
5565,Vianópolis,GO,Região Centro-Oeste,Pequeno I,62,12548.0
5566,Vicentinópolis,GO,Região Centro-Oeste,Pequeno I,62,7371.0
5567,Vila Boa,GO,Região Centro-Oeste,Pequeno I,62,4735.0
5568,Vila Propício,GO,Região Centro-Oeste,Pequeno I,62,5145.0


In [55]:
df_regions.rename(columns={
    'city': 'cidade',
    'state': 'estado',
    'region': 'regiao',
    'size': 'porte_municipio',
    'code': 'codigo_estado',
    'population': 'populacao'
}, inplace=True)

In [56]:
def padronizar_texto(texto):
    if isinstance(texto, str):
        texto = texto.strip()

        texto = unicodedata.normalize('NFKD', texto)

        texto = ''.join(
            c for c in texto
            if not unicodedata.combining(c)
        )

        texto = re.sub(r'[_-]+', ' ', texto)
        texto = re.sub(r'\s+', ' ', texto)

        return texto

    return texto

In [57]:
colunas_texto = [
    'cidade',
    'estado',
    'regiao',
    'porte_municipio'
]

for coluna in colunas_texto:
    df_regions[coluna] = (
        df_regions[coluna].apply(padronizar_texto)
    )

In [58]:
df_regions

,cidade,estado,regiao,porte_municipio,codigo_estado,populacao
0,Alta Floresta Doeste,RO,Regiao Norte,Pequeno II,69,24392.0
1,Ariquemes,RO,Regiao Norte,Medio,69,90353.0
2,Cabixi,RO,Regiao Norte,Pequeno I,69,6313.0
3,Cacoal,RO,Regiao Norte,Medio,69,78574.0
4,Cerejeiras,RO,Regiao Norte,Pequeno I,69,17029.0
...,...,...,...,...,...,...
5565,Vianopolis,GO,Regiao Centro Oeste,Pequeno I,62,12548.0
5566,Vicentinopolis,GO,Regiao Centro Oeste,Pequeno I,62,7371.0
5567,Vila Boa,GO,Regiao Centro Oeste,Pequeno I,62,4735.0
5568,Vila Propicio,GO,Regiao Centro Oeste,Pequeno I,62,5145.0


In [78]:
df_funnel = pd.read_csv(RAW_DIR / 'funnel.csv')
df_stores = pd.read_csv(RAW_DIR / 'stores.csv')


In [79]:
df_funnel.rename(columns={
    'visit_id': 'id_visita',
    'customer_id': 'id_cliente',
    'store_id': 'id_loja',
    'product_id': 'id_produto',
    'visit_page_date': 'data_visita_pagina',
    'add_to_cart_date': 'data_adicao_carrinho',
    'start_checkout_date': 'data_inicio_checkout',
    'finish_checkout_date': 'data_finalizacao_checkout',
    'paid_date': 'data_pagamento',
    'discount': 'desconto'
}, inplace=True)

In [80]:
df_stores.rename(columns={
    'store_name': 'nome_loja',
    'store_cnpj': 'cnpj_loja',
    'store_id': 'id_loja'
}, inplace=True)

In [81]:
df_stores['nome_loja'] = (
    df_stores['nome_loja']
    .apply(padronizar_texto)
)

In [83]:
df_funnel

,id_visita,id_cliente,id_loja,id_produto,data_visita_pagina,data_adicao_carrinho,data_inicio_checkout,data_finalizacao_checkout,data_pagamento,desconto
0,38836BE8B968,73650706FFAE,BF580E604866,produto001,2021-02-19,2021-02-19,NaN,NaN,NaN,-0.34
1,7FB675C2F431,236A739A6063,F79B06D22BC3,produto002,2021-04-20,NaN,NaN,NaN,NaN,-0.38
2,B07A4A5759AC,CF2DFCBE74E3,B06DF477008E,produto003,2021-06-25,NaN,NaN,NaN,NaN,-0.14
3,713A295F9E4C,25504A496AEB,97D85A358174,produto004,2021-06-04,2021-06-04,NaN,NaN,NaN,-0.08
4,D6145FA8A291,05DE6BCDFEC7,BF580E604866,produto005,2021-08-26,NaN,NaN,NaN,NaN,-0.30
...,...,...,...,...,...,...,...,...,...,...
30575,20B6D9D03165,8186B0E0AD47,9EED2B655A9D,produto015,2021-08-16,2021-08-16,NaN,NaN,NaN,-0.10
30576,4B112ED84E7C,028608D204D3,7C759690C7B5,produto024,2021-06-02,2021-06-02,2021-06-09,2021-06-09,2021-06-10,-0.19
30577,9DBCB3AA64C3,9A4B4CE6EF49,D40C552CED19,produto054,2021-07-14,2021-07-14,NaN,NaN,NaN,-0.21
30578,92A2FBB593D3,A5E64FD582D6,449E7D1BBBC9,produto161,2020-12-12,NaN,NaN,NaN,NaN,-0.13


In [84]:
# Dicionário com os DataFrames tratados
dataframes = {
    'customers': df_customers,
    'ibge_genders': df_ibge,
    'products': df_products,
    'regions': df_regions,
    'funnel': df_funnel,
    'stores': df_stores
}

# Exportar arquivos CSV
for nome, dataframe in dataframes.items():
    caminho = TREATED_DIR / f'{nome}.csv'

    dataframe.to_csv(
        caminho,
        index=False,
        encoding='utf-8-sig'
    )

    print(f'Arquivo salvo: {caminho}')

Arquivo salvo: c:\Users\uniao\OneDrive\Área de Trabalho\AI-DRIVEN DATA ENGINEERING\Udemy\Database\projeto-vendas-automotivas\Dataset\tratados\customers.csv
Arquivo salvo: c:\Users\uniao\OneDrive\Área de Trabalho\AI-DRIVEN DATA ENGINEERING\Udemy\Database\projeto-vendas-automotivas\Dataset\tratados\ibge_genders.csv
Arquivo salvo: c:\Users\uniao\OneDrive\Área de Trabalho\AI-DRIVEN DATA ENGINEERING\Udemy\Database\projeto-vendas-automotivas\Dataset\tratados\products.csv
Arquivo salvo: c:\Users\uniao\OneDrive\Área de Trabalho\AI-DRIVEN DATA ENGINEERING\Udemy\Database\projeto-vendas-automotivas\Dataset\tratados\regions.csv
Arquivo salvo: c:\Users\uniao\OneDrive\Área de Trabalho\AI-DRIVEN DATA ENGINEERING\Udemy\Database\projeto-vendas-automotivas\Dataset\tratados\funnel.csv
Arquivo salvo: c:\Users\uniao\OneDrive\Área de Trabalho\AI-DRIVEN DATA ENGINEERING\Udemy\Database\projeto-vendas-automotivas\Dataset\tratados\stores.csv
